# Vienna imagery with STAC and S3

Clip three Sentinel-2 scenes over Vienna, publish the images and a STAC catalogue in `ws-bob/public/`, then read them through STAC and S3.

Use the [demo workspace](quickstart.md#environment-assumptions), or change the names, URLs, and OAuth settings for your deployment.

The scene selection and raster preparation come from the PackageR [Vienna fixture](https://github.com/versioneer-tech/package-r/blob/main/scripts/download_sentinel2.py). It uses scenes from February, May, and August.

The uploaded files become publicly readable through PackageR's [default share](../concepts.md#public-shares).


## 1. Prepare the runtime

Install the dependencies and start Jupyter from the repository root:

```bash
uv sync
uv run jupyter lab
```

Upload from a `ws-bob` Datalab with write access to its storage. The upload cell reads `AWS_ENDPOINT_URL`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`, and `AWS_REGION`. For a local runtime, set those values before starting it.

For S3 downloads, enter the `ws-bob` OAuth client secret at the hidden prompt in step 6. An operator can supply it from `ws-bob-oauth2-client` in the Datalab runtime namespace.

The platform needs a permanent Data address and S3 enabled through the [Gateway settings](installation.md#example-values).


In [ ]:
from __future__ import annotations

import getpass
import hashlib
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path
from urllib.parse import quote, urljoin

import boto3
import pystac
import rasterio
import requests
import stac_geoparquet.arrow
from botocore.config import Config
from botocore.exceptions import ClientError
from IPython.display import Image, display
from pystac.extensions.eo import EOExtension
from pystac.extensions.projection import ProjectionExtension
from rasterio.enums import Resampling
from rasterio.shutil import copy as copy_raster
from rasterio.warp import transform_bounds
from rasterio.windows import Window, from_bounds, transform as window_transform

WORKSPACE = "ws-bob"
BUCKET = "ws-bob"
DATA_URL = "https://ws-bob.lab.develop-v2.eoepca.org"
ISSUER = "https://develop-v2.eoepca.org/iam/auth/realms/eoepca"
S3_REGION = "us-east-1"
DATASET = "vienna-s2l2a-26"
CATALOG_KEY = "public/catalog.parquet"
VIENNA_BBOX = (16.18, 48.11, 16.58, 48.32)
SOURCE_API = "https://earth-search.aws.element84.com/v1"
SOURCE_COLLECTION = "sentinel-2-c1-l2a"
SCENE_IDS = [
    "S2B_T33UXP_20260218T100524_L2A",
    "S2B_T33UXP_20260519T100022_L2A",
    "S2B_T33UWP_20260814T095452_L2A",
]

# Change these public settings for another deployment.
REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "scripts/s3-credentials.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from a checkout of the Workspace repository.")
CREDENTIAL_HELPER = REPO_ROOT / "scripts/s3-credentials.py"
WORK_DIR = Path(tempfile.mkdtemp(prefix="vienna-stac-"))
PUBLIC_DIR = WORK_DIR / "public"
PUBLIC_DIR.mkdir()
DOWNLOAD_DIR = WORK_DIR / "downloads"
DOWNLOAD_DIR.mkdir()

# Optional: reuse the materialized PackageR fixture instead of reading source COGs.
# Set this to the directory that contains the scene directories in that checkout.
FIXTURE_DIR = None

S3_CONFIG = Config(
    signature_version="s3v4",
    s3={"addressing_style": "path"},
    connect_timeout=10,
    read_timeout=120,
    retries={"max_attempts": 3},
)
print({"workspace": WORKSPACE, "bucket": BUCKET, "local_directory": str(WORK_DIR)})

## 2. Select the scenes

Use the fixture's exact scene IDs from Earth Search. They were selected for low reported cloud cover in three time windows.

Crop each raster to the Vienna bounding box and write a Cloud Optimized GeoTIFF with a PNG preview. Record the cropped grid in the STAC metadata. The crop is rectangular; the August scene uses a different tile and can cover a different area.


In [ ]:
response = requests.get(f"{SOURCE_API}/collections/{SOURCE_COLLECTION}", timeout=60)
response.raise_for_status()
source_collection = response.json()
source_license_links = [link for link in source_collection["links"] if link["rel"] == "license"]

source_items = []
for scene_id in SCENE_IDS:
    source_url = f"{SOURCE_API}/collections/{SOURCE_COLLECTION}/items/{scene_id}"
    response = requests.get(source_url, timeout=60)
    response.raise_for_status()
    item = pystac.Item.from_dict(response.json(), migrate=False)
    assert item.id == scene_id
    assert "visual" in item.assets
    source_items.append((item, source_url))

print([
    {"id": item.id, "datetime": item.datetime.isoformat(),
     "cloud_cover": item.properties.get("eo:cloud_cover")}
    for item, _ in source_items
])

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def polygon(bounds):
    west, south, east, north = bounds
    return {
        "type": "Polygon",
        "coordinates": [[[west, south], [east, south], [east, north],
                         [west, north], [west, south]]],
    }


def prepare_image(source_item, destination):
    if FIXTURE_DIR is not None:
        source = Path(FIXTURE_DIR) / source_item.id / "overview.tif"
        if not source.is_file():
            raise FileNotFoundError(source)
        # The fixture already contains a Vienna clip. Leave the source file intact.
        copy_raster(source, destination, driver="COG", compress="DEFLATE")
        return

    temporary = destination.parent / "clipped.tif"
    try:
        with rasterio.Env(
            GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
            CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff",
        ), rasterio.open(source_item.assets["visual"].href) as source:
            bounds = transform_bounds("EPSG:4326", source.crs, *VIENNA_BBOX, densify_pts=21)
            window = from_bounds(*bounds, transform=source.transform)
            window = window.round_offsets().round_lengths().intersection(
                Window(0, 0, source.width, source.height)
            )
            pixels = source.read(window=window)
            profile = source.profile.copy()
            profile.update(
                driver="GTiff", width=pixels.shape[2], height=pixels.shape[1],
                transform=window_transform(window, source.transform),
                tiled=True, blockxsize=512, blockysize=512, compress="DEFLATE",
            )
            with rasterio.open(temporary, "w", **profile) as target:
                target.write(pixels)
                target.colorinterp = source.colorinterp
            copy_raster(temporary, destination, driver="COG", compress="DEFLATE",
                        overview_resampling="AVERAGE")
    finally:
        temporary.unlink(missing_ok=True)


items = []
for source_item, source_url in source_items:
    directory = PUBLIC_DIR / DATASET / source_item.id
    directory.mkdir(parents=True, exist_ok=True)
    image_path = directory / "overview.tif"
    preview_path = directory / "preview.png"
    prepare_image(source_item, image_path)

    with rasterio.open(image_path) as image:
        assert image.count == 3 and image.dtypes == ("uint8",) * 3
        bounds = list(transform_bounds(image.crs, "EPSG:4326", *image.bounds, densify_pts=21))
        item = pystac.Item(
            id=source_item.id, geometry=polygon(bounds), bbox=bounds,
            datetime=source_item.datetime,
            properties={"title": "Vienna colour image", "license": source_collection["license"]},
        )
        for link in source_license_links:
            item.add_link(pystac.Link.from_dict(link))
        projection = {
            "code": f"EPSG:{image.crs.to_epsg()}",
            "shape": [image.height, image.width],
            "transform": list(image.transform)[:6],
        }
        cloud_cover = source_item.properties.get("eo:cloud_cover")
        if cloud_cover is not None:
            EOExtension.ext(item, add_if_missing=True).apply(cloud_cover=cloud_cover)
        item.add_link(pystac.Link(rel="derived_from", target=source_url,
                                 media_type="application/geo+json"))
        scale = min(1, 640 / max(image.width, image.height))
        preview = image.read(
            out_shape=(3, max(1, round(image.height * scale)), max(1, round(image.width * scale))),
            resampling=Resampling.bilinear,
        )

    with rasterio.open(preview_path, "w", driver="PNG", count=3, dtype="uint8",
                       height=preview.shape[1], width=preview.shape[2]) as target:
        target.write(preview)
    item.add_asset("overview", pystac.Asset(
        href=image_path.relative_to(PUBLIC_DIR).as_posix(),
        media_type="image/tiff; application=geotiff; profile=cloud-optimized",
        title="Vienna colour COG", roles=["visual"],
    ))
    ProjectionExtension.ext(item.assets["overview"], add_if_missing=True).apply(**projection)
    item.add_asset("preview", pystac.Asset(
        href=preview_path.relative_to(PUBLIC_DIR).as_posix(),
        media_type="image/png", title="Vienna preview", roles=["thumbnail"],
    ))
    items.append(item)
    print({"id": item.id, "bbox": item.bbox, "image_bytes": image_path.stat().st_size})
    display(Image(filename=str(preview_path)))

## 3. Write the STAC catalogue

The public share reads `public/catalog.parquet`. Asset paths must be relative to `public`, such as `vienna-s2l2a-26/<scene-id>/overview.tif`. PackageR turns those paths into public download links in returned STAC Items.

The catalogue records the source scene links, license, and cropped grid. It contains colour imagery and previews; use the original reflectance bands for numerical analysis.


In [ ]:
catalog_path = PUBLIC_DIR / "catalog.parquet"
table = stac_geoparquet.arrow.parse_stac_items_to_arrow([item.to_dict() for item in items]).read_all()
stac_geoparquet.arrow.to_parquet(table, catalog_path)

# Include only the catalog and the assets named in its Items.
publish_files = {CATALOG_KEY: catalog_path}
for item in items:
    for asset in item.assets.values():
        publish_files[f"public/{asset.href}"] = PUBLIC_DIR / asset.href
expected_hashes = {key: sha256(path) for key, path in publish_files.items()}
print({"catalog_rows": table.num_rows, "files": len(publish_files),
       "total_bytes": sum(path.stat().st_size for path in publish_files.values())})

## 4. Upload to the bucket

Upload with the Datalab's backing storage credentials, not the Data S3 endpoint used for downloads later.

Existing files are checked by SHA-256. Matching files are skipped; different content at the same key stops the upload. If `public/catalog.parquet` already describes another dataset, merge the catalogues first. Upload the catalogue after its assets.


In [ ]:
required = ["AWS_ENDPOINT_URL", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError("Set the ws-bob backing storage settings: " + ", ".join(missing))
storage_endpoint = os.environ["AWS_ENDPOINT_URL"].rstrip("/")
if storage_endpoint == DATA_URL.rstrip("/"):
    raise ValueError("Upload needs the backing storage endpoint; the Data endpoint is read only.")

storage = boto3.client(
    "s3", endpoint_url=storage_endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=os.getenv("AWS_SESSION_TOKEN"),
    region_name=os.getenv("AWS_REGION", S3_REGION), config=S3_CONFIG,
)


def object_sha256(client, key):
    response = client.get_object(Bucket=BUCKET, Key=key)
    digest = hashlib.sha256()
    with response["Body"] as body:
        for chunk in iter(lambda: body.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


existing = set()
for key in publish_files:
    try:
        storage.head_object(Bucket=BUCKET, Key=key)
    except ClientError as error:
        if error.response["Error"]["Code"] not in ("404", "NoSuchKey", "NotFound"):
            raise
    else:
        if object_sha256(storage, key) != expected_hashes[key]:
            raise RuntimeError(f"Existing object has different content: s3://{BUCKET}/{key}")
        existing.add(key)

for key in sorted(publish_files, key=lambda key: key == CATALOG_KEY):
    if key in existing:
        continue
    path = publish_files[key]
    media_type = (
        "application/vnd.apache.parquet" if path.suffix == ".parquet" else
        "image/png" if path.suffix == ".png" else
        "image/tiff; application=geotiff; profile=cloud-optimized"
    )
    with path.open("rb") as body:
        storage.put_object(
            Bucket=BUCKET, Key=key, Body=body, ContentType=media_type,
            Metadata={"sha256": expected_hashes[key]}, IfNoneMatch="*",
        )
    assert object_sha256(storage, key) == expected_hashes[key]

print({"uploaded": len(publish_files) - len(existing), "already_present": len(existing),
       "public_share": f"{DATA_URL}/share/public/"})

## 5. Browse STAC and download an image

Open the [public share](https://ws-bob.lab.develop-v2.eoepca.org/share/public/) or its catalogue:

[STAC Collection](https://ws-bob.lab.develop-v2.eoepca.org/api/public/catalog/public)

Download the published files:

| Scene date | COG image | PNG preview |
| --- | --- | --- |
| 18 February 2026 | [Download](https://ws-bob.lab.develop-v2.eoepca.org/api/public/share/public/vienna-s2l2a-26/S2B_T33UXP_20260218T100524_L2A/overview.tif?presign=true&follow=true) | [View](https://ws-bob.lab.develop-v2.eoepca.org/api/public/share/public/vienna-s2l2a-26/S2B_T33UXP_20260218T100524_L2A/preview.png?presign=true&follow=true) |
| 19 May 2026 | [Download](https://ws-bob.lab.develop-v2.eoepca.org/api/public/share/public/vienna-s2l2a-26/S2B_T33UXP_20260519T100022_L2A/overview.tif?presign=true&follow=true) | [View](https://ws-bob.lab.develop-v2.eoepca.org/api/public/share/public/vienna-s2l2a-26/S2B_T33UXP_20260519T100022_L2A/preview.png?presign=true&follow=true) |
| 14 August 2026 | [Download](https://ws-bob.lab.develop-v2.eoepca.org/api/public/share/public/vienna-s2l2a-26/S2B_T33UWP_20260814T095452_L2A/overview.tif?presign=true&follow=true) | [View](https://ws-bob.lab.develop-v2.eoepca.org/api/public/share/public/vienna-s2l2a-26/S2B_T33UWP_20260814T095452_L2A/preview.png?presign=true&follow=true) |

Follow the Collection's `rel=item` links to the Items and their assets. The catalogue has no `/search` endpoint. The cells below check its records and download links.


In [ ]:
catalog_url = f"{DATA_URL}/api/public/catalog/public"
response = requests.get(catalog_url, timeout=120)
response.raise_for_status()
collection = response.json()
assert collection["type"] == "Collection" and collection["id"] == "public"
item_links = [link for link in collection["links"] if link["rel"] == "item"]
served_items = []
for link in item_links:
    response = requests.get(urljoin(catalog_url, link["href"]), timeout=120)
    response.raise_for_status()
    item = response.json()
    assert item["type"] == "Feature" and item["geometry"] and item["bbox"]
    for asset in item["assets"].values():
        assert asset["href"].startswith(f"{DATA_URL}/api/public/share/public/")
    served_items.append(item)
assert {item["id"] for item in served_items} == set(SCENE_IDS)
served_item = next(item for item in served_items if item["id"] == SCENE_IDS[0])

print({"collection": collection["id"], "extent": collection["extent"],
       "items": [item["id"] for item in served_items],
       "first_item": next(link["href"] for link in served_item["links"] if link["rel"] == "self")})

In [ ]:
def download_public(url, destination):
    # The asset URL redirects to a short-lived backing storage URL.
    # Keep that signed URL out of output and files.
    with requests.get(url, stream=True, timeout=(10, 120)) as response:
        if response.status_code != 200:
            raise RuntimeError(f"Public download failed (HTTP {response.status_code}).")
        with destination.open("wb") as output:
            for chunk in response.iter_content(1024 * 1024):
                output.write(chunk)


first_image_key = f"public/{DATASET}/{served_item['id']}/overview.tif"
public_download = DOWNLOAD_DIR / "public-overview.tif"
download_public(served_item["assets"]["overview"]["href"], public_download)
assert sha256(public_download) == expected_hashes[first_image_key]
public_preview = DOWNLOAD_DIR / "public-preview.png"
download_public(served_item["assets"]["preview"]["href"], public_preview)
assert sha256(public_preview) == expected_hashes[f"public/{DATASET}/{served_item['id']}/preview.png"]
display(Image(filename=str(public_preview)))
print({"download": str(public_download), "sha256_matches": True})

For a terminal download, use the public-share route. `curl --location` follows its redirect to storage:

```bash
curl --fail --location --output vienna-overview.tif \
  'https://ws-bob.lab.develop-v2.eoepca.org/api/public/share/public/vienna-s2l2a-26/S2B_T33UXP_20260218T100524_L2A/overview.tif?presign=true&follow=true'
```

Share the public-share link. The final storage URL expires.


## 6. Get S3 credentials

Enter the `ws-bob` client secret at the hidden prompt. The cell runs `scripts/s3-credentials.py` and keeps the returned credentials in memory. Rerun it when the token expires.

See [S3 clients](../concepts.md#s3-clients) for the signing fields and required token claims.


In [ ]:
client_secret = getpass.getpass("ws-bob OAuth client secret: ")
try:
    result = subprocess.run(
        [sys.executable, str(CREDENTIAL_HELPER), "--tenant", WORKSPACE, "--issuer", ISSUER],
        env={"CLIENT_SECRET": client_secret},
        capture_output=True, text=True, check=False,
    )
finally:
    del client_secret
if result.returncode:
    raise RuntimeError("Cannot get S3 credentials. Check the client secret and issuer.")
s3_credentials = json.loads(result.stdout)
del result
gateway = boto3.client(
    "s3", endpoint_url=DATA_URL, region_name=S3_REGION, config=S3_CONFIG,
    aws_access_key_id=s3_credentials["AccessKeyId"],
    aws_secret_access_key=s3_credentials["SecretAccessKey"],
    aws_session_token=s3_credentials["SessionToken"],
)
print({"endpoint": DATA_URL, "credentials_expire": s3_credentials["Expiration"]})
del s3_credentials

## 7. Read through S3

Read bucket `ws-bob` with prefix `public/vienna-s2l2a-26/` using path-style requests. List files, inspect an object, check a [range read](../concepts.md#s3-clients), and download with `get_object`.


In [ ]:
buckets = gateway.list_buckets()["Buckets"]
assert BUCKET in {bucket["Name"] for bucket in buckets}
listed = []
for page in gateway.get_paginator("list_objects_v2").paginate(
    Bucket=BUCKET, Prefix=f"public/{DATASET}/"
):
    listed.extend(page.get("Contents", []))
assert set(publish_files) - {CATALOG_KEY} <= {entry["Key"] for entry in listed}
print({"buckets": [bucket["Name"] for bucket in buckets],
       "objects": [{"key": entry["Key"], "bytes": entry["Size"]} for entry in listed]})

head = gateway.head_object(Bucket=BUCKET, Key=first_image_key)
assert head["ContentLength"] == publish_files[first_image_key].stat().st_size
response = gateway.get_object(Bucket=BUCKET, Key=first_image_key, Range="bytes=0-1023")
with response["Body"] as body:
    first_bytes = body.read()
assert response["ResponseMetadata"]["HTTPStatusCode"] in (200, 206)
assert response["ContentRange"] == f"bytes 0-1023/{head['ContentLength']}"
with publish_files[first_image_key].open("rb") as source:
    assert first_bytes == source.read(1024)
print({"head_bytes": head["ContentLength"], "range_bytes": len(first_bytes),
       "range_matches": True})

In [ ]:
for key in publish_files:
    destination = DOWNLOAD_DIR / "s3" / key.removeprefix("public/")
    destination.parent.mkdir(parents=True, exist_ok=True)
    response = gateway.get_object(Bucket=BUCKET, Key=key)
    with response["Body"] as body, destination.open("wb") as output:
        for chunk in iter(lambda: body.read(1024 * 1024), b""):
            output.write(chunk)
    assert sha256(destination) == expected_hashes[key]
print({"verified_s3_downloads": len(publish_files),
       "directory": str(DOWNLOAD_DIR / "s3"), "sha256_matches": True})

## 8. Download with the AWS CLI

Run these commands from the repository root. They create a temporary AWS profile whose `credential_process` gets new tokens when needed. Enter the client secret at the hidden prompt.

```bash
aws_config_dir="$(mktemp -d /tmp/workspace-aws-XXXXXX)"
export AWS_CONFIG_FILE="$aws_config_dir/config"
export AWS_SHARED_CREDENTIALS_FILE="$aws_config_dir/credentials"
touch "$AWS_CONFIG_FILE" "$AWS_SHARED_CREDENTIALS_FILE"
chmod 600 "$AWS_CONFIG_FILE" "$AWS_SHARED_CREDENTIALS_FILE"
unset AWS_ACCESS_KEY_ID AWS_SECRET_ACCESS_KEY AWS_SESSION_TOKEN AWS_SECURITY_TOKEN
unset AWS_ENDPOINT_URL AWS_ENDPOINT_URL_S3 AWS_REGION AWS_DEFAULT_REGION AWS_PROFILE
export AWS_EC2_METADATA_DISABLED=true
export AWS_REQUEST_CHECKSUM_CALCULATION=when_required
export AWS_RESPONSE_CHECKSUM_VALIDATION=when_required
export AWS_DEFAULT_OUTPUT=json

read -r -s -p 'ws-bob OAuth client secret: ' CLIENT_SECRET
printf '\n'
export CLIENT_SECRET
aws configure set region us-east-1 --profile ws-bob
aws configure set endpoint_url https://ws-bob.lab.develop-v2.eoepca.org --profile ws-bob
aws configure set s3.addressing_style path --profile ws-bob
aws configure set credential_process \
  "python3 \"$PWD/scripts/s3-credentials.py\" --tenant ws-bob --issuer https://develop-v2.eoepca.org/iam/auth/realms/eoepca" \
  --profile ws-bob

aws --profile ws-bob s3 ls s3://ws-bob/public/vienna-s2l2a-26/ --recursive
aws --profile ws-bob s3api head-object \
  --bucket ws-bob \
  --key public/vienna-s2l2a-26/S2B_T33UXP_20260218T100524_L2A/overview.tif
aws --profile ws-bob s3api get-object \
  --bucket ws-bob \
  --key public/vienna-s2l2a-26/S2B_T33UXP_20260218T100524_L2A/overview.tif \
  vienna-overview.tif
```

Keep `CLIENT_SECRET` set while the CLI runs. Avoid displaying the helper's output: it contains the access token. To share an image, use the public download link from step 5.

Remove the temporary profile and clear the secret when finished:

```bash
unset CLIENT_SECRET AWS_CONFIG_FILE AWS_SHARED_CREDENTIALS_FILE
rm -f "$aws_config_dir/config" "$aws_config_dir/credentials"
rmdir "$aws_config_dir"
unset aws_config_dir
```

The published files stay in `ws-bob/public`. Local files are in the temporary directory printed in step 1.

The published example was checked on 23 October 2026. The checks covered the three STAC Items, public image downloads, S3 listing, HEAD, and byte-range GET. All seven S3 downloads matched their local SHA-256 checksums.
